In [48]:
# Import libraries
using Turing
using LinearAlgebra
using Distributions
using MultivariateStats
import MultivariateStats: reconstruct
using GaussianProcesses
using Optim
using StatsBase
using Statistics
using Suppressor
using JLD2
using CSV
using DataFrames, DataFramesMeta
using SplitApplyCombine
using KernelFunctions
using MCMCChains
using QuantileRegressions
using PyCall
using PyPlot
using Printf
import PyCall.pyfunction

include("../Utils/trapezoidal.jl")

include("../Utils/scale_utils.jl")
using .ScaleUtils

include("../HPC_Bayesian_Calibration/amery_calibration.jl")
using .amery_sequential_calibration 

include("../Utils/gp_utils.jl")
using .GPUtils

scipy = pyimport("scipy")
np = pyimport("numpy")


# Set a seed for reproducibility
using Random
Random.seed!(11);

In [49]:
# Managing directories for saveouts during the calibration processes

# Should be a string: "/home/zzerbe/ProjectFolder/BrookhavenCode/ProjectNotebooks/OUTPUT_R_1"
saveout_directory = "../Data/Local_Calibration_Chains"

#Make a directory for saving the returned Markov Chains
if !isdir("$(saveout_directory)/mcmc_chains_dir")
        mkdir("$(saveout_directory)/mcmc_chains_dir")
        chains_dir = "$(saveout_directory)/mcmc_chains_dir"
else
        chains_dir = "$(saveout_directory)/mcmc_chains_dir"
end

#Make a directory for saving the quantities generated during sampling (Log-priors, log-likelihood, log-posteriors, emulator outputs)
if !isdir("$(saveout_directory)/generated_quantities_dir")
        mkdir("$(saveout_directory)/generated_quantities_dir")
        gq_dir = "$(saveout_directory)/generated_quantities_dir"
else
        gq_dir = "$(saveout_directory)/generated_quantities_dir"
end


"../Data/Local_Calibration_Chains/generated_quantities_dir"

In [50]:

# "Global" VARIABLES

future_years = [2030, 2045, 2060, 2075, 2090, 2105, 2120, 2135, 2150, 2165, 2180, 2195, 2210, 2225, 2240, 2255, 2270, 2285, 2300]

# Minimum and maximum values of the datasets for the three observables used to calibrate the 2015 priors 
#(Calving front, grounding line, surface mass balance), used for scaling purposes

Yc_mins  = [-96.34022417]
Yc_maxs  = [74.9385927]
Yg_mins  = [-21.10306973]
Yg_maxs  = [19.2289032]
Ysmb_mins  = [-23.91989209]
Ysmb_maxs  = [21.04901554]




1-element Vector{Float64}:
 21.04901554

In [51]:
        #RELX Ensemble Mali input parameters
        X_raw = CSV.read("../Data/Training_Data/Amery_GP_Emulator_RELX_Ensemble_Filtered.csv", DataFrame);
        # 1) Grab all column‐names as Symbols
        cols = Symbol.(names(X_raw))

        # 2) Remove the index‐column symbol
        cols = filter(c -> c != :Column1, cols)

        # 3) Now call get_scaled_matrix on the remaining columns
        X_scaled_t, X_scalers, X_min_dict, X_max_dict = ScaleUtils.get_scaled_matrix(X_raw, cols)

        #####################################################################################################################

        # Emulators trained on historical observations 
        cal_path = "../Data/calving_front_gap_15_emulators.jld2"
        grd_path = "../Data/grounding_line_gap_15_emulators.jld2"
        smb_path = "../Data/mass_balance_gap_15_emulators.jld2"

        # Load emulator dictionaries from disk:
        gp_cal_dict  = GPUtils.load_emulators(cal_path)
        gp_grd_dict  = GPUtils.load_emulators(grd_path)
        gp_smb_dict = GPUtils.load_emulators(smb_path)


        #####################################################################################################################

        # Cumulative Grounded Mass Change minimums, maximums, and emulators
        y_mass_mins, y_mass_maxs = JLD2.load("../Data/Training_Data/grd_mass_training_mins_maxs.jld2", "y_grd_mass_mins", "y_grd_mass_maxs")

        #sort the mins and maxs by year, collect into vectors
        ordered_mass_mins = collect(values(sort(y_mass_mins)))
        ordered_mass_maxs = collect(values(sort(y_mass_maxs)))

        # Load cumulative variance data
        mass_uncertainties = CSV.read("../Data/Training_Data/sigma_grd_mass_change.csv", DataFrame);

        #load scalar cumulative grounded mass change emulators
        gps_mass_all = GPUtils.load_emulators("../Data/Grd_mass_change_gap_15_emulators.jld2");



        #####################################################################################################################

        #subset the relx emulators for just the 2015 instances
        gps_cal  = [gp_cal_dict[2015]]
        gps_grd  = [gp_grd_dict[2015]]
        gps_smb = [gp_mass_dict[2015]]

        #Just renamed these for the main function cball
        Ymass_mins = ordered_mass_mins
        Ymass_maxs = ordered_mass_maxs;

In [52]:
 #Load future observations
        fut_obs_path = "../Data/Future_Observation_Data/Official_Constraining_Observations.jld2"
        
        #Size 100x19
        all_future_obs = JLD2.load(fut_obs_path, "Y_grd_mass_obs") 

        # Note that our project calibrated on realization_id's 1-100
        realization_id = 1

        #Subsetting the matrix of future observations according to the Realization ID, yielding a single future trajectory of 19 years worth of             #future grounded massume change observations
        future_obs = all_future_obs[realization_id,:]

        for i in 1:length(future_years)
    
            years_selected = future_years[1:i]
            #Nothing returned by this function
            amery_sequential_calibration.calibration_func(future_obs, 
                                                            future_years,
                                                            years_selected,
                                                            chains_dir,
                                                            gq_dir,
                                                            X_min_dict, 
                                                            X_max_dict,
                                                            Yc_mins,
                                                            Yc_maxs,
                                                            Yg_mins,
                                                            Yg_maxs,
                                                            Ysmb_mins,
                                                            Ysmb_maxs,
                                                            Ymass_mins,
                                                            Ymass_maxs,
                                                            mass_uncertainties,
                                                            gps_cal,
                                                            gps_grd,
                                                            gps_smb,
                                                            gps_mass_all )
            
        end
                                                

[1]


┌ Info: Found initial step size
└   ϵ = 0.8
Sampling (Chain 1 of 1): 100%|██████████████████████████| Time: 0:00:03


Sampled and saved with 1 years of future constraining data
[1, 2]


┌ Info: Found initial step size
└   ϵ = 0.4
Sampling (Chain 1 of 1): 100%|██████████████████████████| Time: 0:00:04


Sampled and saved with 2 years of future constraining data
[1, 2, 3]


┌ Info: Found initial step size
└   ϵ = 0.28828125000000004
Sampling (Chain 1 of 1): 100%|██████████████████████████| Time: 0:00:06


Sampled and saved with 3 years of future constraining data
[1, 2, 3, 4]


┌ Info: Found initial step size
└   ϵ = 0.4
Sampling (Chain 1 of 1): 100%|██████████████████████████| Time: 0:00:02


LoadError: InterruptException: